<a href="https://colab.research.google.com/github/BraedynL0530/autocaptcha/blob/master/SuperCoolCaptchaBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y tesseract-ocr
!pip install ultralytics fastapi uvicorn python-multipart pytesseract nest_asyncio

import asyncio
import time
from fastapi import FastApi, UploadFile, File
from ultralytics import  YOLO
import cv2
import numpy as npp
import pytessesract
import nest_asyncio
import uvicorn
app = FastApi()

model = YOLO('yolov11n.pt')
@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    contents = await file.read()
    nparr = np.frombuffer(contents, npuint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    results = model(img)

    detectedBoxes = []
    for box in results[0].boxes:
        cx, cy, w, h = box.xywh.cpu().tolist()[0]
        classId = int(box.cls.cpu().tolist()[0])
        label = model.names[classId]
        detectedBoxes.append({
            "cords": [cx,cy,w,h],
            "label": label
        })

        captchaPrompt = pytessesract.image_to_string(img, lang='eng').lower()
        return{
            "prompt":captchaPrompt,
            "boxes": detectedBoxes
        }

nest_asyncio.apply()

async def start_uvicorn():
    confing = uvicorn.Config(app,host="127.0.0.1",port=8000, loop ="asyncio")
    server = uvicorn.Server(confing)
    await server.serve()

loop = asyncio.get_event_loop()
loop.create_task(start_uvicorn())
time.sleep(4)

!ssh -o StrictHostKeyChecking=no -R 80:localhost:8000 serveo.net

